# 04 Redis 功能与淘汰

## 八股问答

### 1. 过期删除和内存淘汰的区别？

过期删除针对**已经设置了 TTL、到期失效的 key**，目标是把不再使用的数据从逻辑上清理。

内存淘汰针对**内存达到 `maxmemory` 上限**时的容量压力，目标是根据策略选择牺牲哪些 key，即使它们还没有过期。

```text
过期删除：key 到了 TTL，系统想办法清理它
内存淘汰：内存满了，系统按策略选一些 key 腾出空间
```

两者可以同时发生：有过期 key 时执行过期删除；内存仍不足时，再执行淘汰策略。

### 2. 过期删除策略？

Redis 的过期删除采用**惰性删除 + 周期性删除**结合。

惰性删除：

- 访问 key 时才检查是否过期。
- 过期则删除并返回空，优点是不额外扫描所有 key。
- 缺点是没有访问的过期 key 会一直占用内存。

周期性删除：

- 后台定时任务抽样检查一部分设置了 TTL 的 key。
- 发现过期就删除，并控制每次扫描的时间和次数，避免阻塞主线程。
- 若过期 key 占比仍高，会继续多轮清理；占比下降后降低频率。

为什么不只用定时器逐个删除：key 数量大时维护定时器占用内存，逐个触发也会造成 CPU 抖动。
为什么不只用惰性删除：无人访问的过期数据会留在内存里。

### 3. 内存淘汰策略（Redis 缓存满了怎么办）？

通过 `maxmemory` 设置内存上限，再用 `maxmemory-policy` 选择策略：

| 策略 | 含义 | 是否可能丢失未过期数据 |
| --- | --- | --- |
| `noeviction` | 内存满时拒绝写命令 | 不会主动淘汰，但写入失败 |
| `volatile-lru` | 只淘汰设置了 TTL 的 key，按 LRU | 可能淘汰有效 key |
| `allkeys-lru` | 淘汰所有 key，按 LRU | 会 |
| `volatile-lfu` | 只淘汰设置了 TTL 的 key，按 LFU | 可能淘汰有效 key |
| `allkeys-lfu` | 淘汰所有 key，按 LFU | 会 |
| `volatile-random` | 只随机淘汰设置了 TTL 的 key | 可能淘汰有效 key |
| `allkeys-random` | 随机淘汰所有 key | 会 |
| `volatile-ttl` | 优先淘汰 TTL 更短的 key | 可能淘汰有效 key |

常用建议：

- 缓存场景通常使用 `allkeys-lru` 或 `allkeys-lfu`。
- 需要保证某些数据不丢失时，不要把 Redis 当作唯一数据源。
- 监控 `maxmemory`、淘汰次数、命中率，避免淘汰风暴。
- 不同业务可以分实例，避免大流量业务挤占其他业务内存。

### 4. 讲一下 LRU 和 LFU 算法？

**LRU（Least Recently Used）**：优先淘汰最久没被访问的 key。

适合访问模式比较稳定的缓存，例如“最近看过的商品”。但一次性批量访问冷数据时，可能把热点数据挤出去。

Redis 的 LRU 不是维护全局精确的链表，而是**近似 LRU**：

- 给每个 key 记录最近访问时间。
- 淘汰时随机采样一部分 key，例如 `maxmemory-samples 5`。
- 从样本中选择最久未访问的 key 删除。

好处是内存开销小，坏处是结果近似，不是绝对精确的 LRU。

**LFU（Least Frequently Used）**：优先淘汰访问频率最低的 key。

Redis 4.0 后支持 LFU，使用一个计数器记录访问频率，并考虑时间衰减，避免长期不访问的 key 一直占着位置。

适合“少数 key 高频访问”的场景，例如热搜词。比起 LRU，它能更好地抵抗一次批量访问导致的缓存污染。

选择时看业务：命中模式更接近“最近使用”就用 LRU；更接近“某几个 key 一直被高频使用”就用 LFU。


## 面试话术

“过期删除解决的是 TTL 到期数据，内存淘汰解决的是内存满的问题。Redis 用惰性删除和周期性抽样删除处理过期 key；
内存满了按 `maxmemory-policy` 淘汰，缓存场景常用 `allkeys-lru` 或 `allkeys-lfu`。
Redis 的 LRU 是近似 LRU，通过随机采样减少内存开销；LFU 记录访问频率并有时间衰减，更适合热点集中的场景。
我会根据业务命中模式选择策略，同时监控淘汰次数和命中率，避免把 Redis 当成唯一数据源。”


## 自查清单

- [ ] 能区分过期删除和内存淘汰
- [ ] 能解释惰性删除和周期性删除的配合
- [ ] 能说出常用 `maxmemory-policy` 及适用场景
- [ ] 能说明近似 LRU 和 LFU 的差异
